# 🎲 Procedural NPC Generator MVP — Qwen3-TTS

This notebook uses **Qwen3-TTS VoiceDesign** to procedurally generate endless unique NPCs based on genres, traits, and random seeds. It constructs a backstory, a voice description prompt, and a signature line, then renders the audio deterministically.

In [ ]:
!pip install -q qwen-tts soundfile

import os
import gc
import torch
import random
import hashlib
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/procedural_npcs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GENRE_POOLS = {
    "fantasy_tavern": {
        "name_first": ["Aldric", "Mira", "Gareth", "Lyssa", "Theron", "Bryn", "Cael", "Vesper", "Rowan", "Isolde"],
        "name_last": ["Stonebrew", "Ashwood", "Ironfist", "Fairweather", "Coldwater", "Grimshaw", "Brightmoor", "Duskhollow"],
        "occupation": ["innkeeper", "traveling merchant", "retired soldier", "wandering bard", "local farmer", "blacksmith", "healer", "town guard"],
        "personality_traits": ["gruff but fair", "endlessly cheerful", "suspicious of strangers", "deeply nostalgic", "quietly melancholic", "aggressively friendly", "secretive", "boastful"],
        "voice_qualities": ["warm and gravelly", "high-pitched and quick", "slow and measured", "raspy from years of hard living", "melodic and gentle", "booming and theatrical"],
        "accent_hints": ["slight northern burr", "crisp southern diction", "rural drawl", "city-educated precision", "lilting musical quality"],
        "signature_lines": [
            "I've been in this town thirty years and I've seen stranger things than you.",
            "The ale here is terrible but it's the only ale there is, so here we are.",
            "Don't touch that. I mean it. That's not for touching.",
            "My father's father built this place with his bare hands. I'm just trying not to burn it down.",
            "Last adventurer who came through here looking for glory never came back. Third shelf if you're interested."
        ]
    },
    "sci_fi_colony": {
        "name_first": ["Kira", "Zhen", "Orion", "Lyra", "Dax", "Sable", "Vex", "Nova", "Cass", "Ryn"],
        "name_last": ["Chen-7", "Vasquez", "al-Rashid", "Okonkwo", "Lindqvist", "Moreau", "Tanaka", "Singh"],
        "occupation": ["colony engineer", "terraforming specialist", "med-tech", "security officer", "supply coordinator", "comms operator", "xenobiologist", "shuttle pilot"],
        "personality_traits": ["quietly exhausted", "relentlessly optimistic", "bureaucratically frustrated", "professionally paranoid", "dangerously curious", "deeply homesick", "grimly pragmatic"],
        "voice_qualities": ["flat and efficient", "warm despite the circumstances", "clipped military precision", "slow and deliberate", "fast-talking under stress"],
        "accent_hints": ["neutral corporate diction", "slight orbital colony accent", "Earth-nostalgic cadence", "technical jargon-heavy"],
        "signature_lines": [
            "The recyclers are running at sixty percent. I've filed seventeen requests. Nobody cares.",
            "I didn't sign up for hero work. I signed up for my student loans. Different motivations.",
            "Three more years on this rock and I'm taking the first transport back to somewhere with actual weather.",
            "I've run the calculations four times. The fourth time was hoping for a different answer.",
            "Last person who called this a routine mission is no longer with us. Just saying."
        ]
    },
    "horror_mansion": {
        "name_first": ["Edmund", "Elara", "Victor", "Constance", "Dorian", "Agnes", "Silas", "Harriet"],
        "name_last": ["Blackwell", "Ashmore", "Graves", "Whitmore", "Ravenscroft", "Duskwood", "Holloway"],
        "occupation": ["butler", "groundskeeper", "housemaid", "cook", "governess", "mysterious guest", "estate solicitor", "long-term resident"],
        "personality_traits": ["unnervingly polite", "cryptically helpful", "vaguely threatening", "obliviously cheerful", "deeply in denial", "resigned to something awful"],
        "voice_qualities": ["precisely formal", "softly ominous", "too cheerful", "slightly hollow", "warmly wrong"],
        "accent_hints": ["crisp Victorian formality", "regional country accent", "educated but strained"],
        "signature_lines": [
            "The master requests you do not go to the east wing after midnight. He requests it quite strongly.",
            "The previous guests also asked about those noises. They were... relocated.",
            "What a lovely family portrait. The family stopped sitting for portraits in 1887. I'm not sure why.",
            "Cook says dinner will be served at seven. Cook says a great many things. Not all of them make sense.",
            "The dog doesn't usually bark at guests. This is the first time I've seen the dog afraid."
        ]
    }
}

def generate_npc(genre, seed_int):
    """Generate a deterministic NPC from a genre and integer seed."""
    rng = random.Random(seed_int)
    pool = GENRE_POOLS[genre]
    name = f"{rng.choice(pool['name_first'])} {rng.choice(pool['name_last'])}"
    occupation = rng.choice(pool['occupation'])
    trait = rng.choice(pool['personality_traits'])
    voice_q = rng.choice(pool['voice_qualities'])
    accent = rng.choice(pool['accent_hints'])
    line = rng.choice(pool['signature_lines'])
    
    voice_prompt = f"A {occupation} — {trait}, {voice_q}, {accent}"
    backstory = f"{name} is a {occupation}. Known for being {trait}."
    
    return {
        "name": name,
        "occupation": occupation,
        "trait": trait,
        "voice_prompt": voice_prompt,
        "backstory": backstory,
        "signature_line": line,
        "genre": genre,
        "seed": seed_int,
        "voice_q": voice_q,
        "accent": accent
    }

In [ ]:
print("Loading Qwen3-TTS VoiceDesign model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
vd_model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

In [ ]:
def print_voice_card(npc):
    print("\n" + "╔" + "═"*42 + "╗")
    print(f"║  NPC: {npc['name'][:30]:<30} (Seed #{npc['seed']})\n"
          f"║  Genre: {npc['genre'][:31]:<31}\n"
          f"║  Role: {npc['occupation'][:32]:<32}\n"
          f"║  Trait: {npc['trait'][:31]:<31}\n"
          f"║  Voice: {npc['voice_q'][:31]:<31}\n"
          f"║         {npc['accent'][:31]:<31}")
    print("╚" + "═"*42 + "╝")

def generate_and_play_npc(npc):
    print_voice_card(npc)
    print(f"> \"{npc['signature_line']}\"")
    
    try:
        audio, sr = vd_model.generate_voice_design(
            text=npc['signature_line'],
            language="English",
            instruct=npc['voice_prompt']
        )
        
        audio_np = audio.squeeze().cpu().numpy()
        name_clean = npc['name'].replace(' ', '_').lower()
        filename = f"npc_{name_clean}_{npc['seed']}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        sf.write(filepath, audio_np, sr)
        display(Audio(filepath))
        return filepath
        
    except Exception as e:
        print(f"Error generating NPC: {e}")
        return None

print("=== Single NPC Generation ===")
GENRE = "fantasy_tavern"
SEED = 42
npc_single = generate_npc(GENRE, SEED)
generate_and_play_npc(npc_single)

In [ ]:
print("=== Batch NPC Generation ===")
GENRE = "fantasy_tavern"
NUM_NPCS = 5

for seed in range(1, NUM_NPCS + 1):
    npc = generate_npc(GENRE, seed)
    generate_and_play_npc(npc)

print(f"\nGenerated {NUM_NPCS} unique NPCs — all from the same genre, all distinct.")

In [ ]:
print("=== Multi-Genre Showroom ===")
print("Comparing Seed #7 across all genres:\n")

for genre in GENRE_POOLS.keys():
    npc = generate_npc(genre, 7)
    generate_and_play_npc(npc)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/procedural_npcs", 'zip', OUTPUT_DIR)
files.download("/content/procedural_npcs.zip")